# Wall surface coefficients (script)

Script version of the notebook on the documentation website, without the theory and schematics. Set `XFOIL_BIN` in the parameters cell to the path of your XFOIL binary, then run all cells.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Input parameters

In [ ]:
AIRFOIL = "NACA 0012"   # "NACA xxxx" or the path to a coordinate file (.dat)
RE      = 1.0e6         # chord Reynolds number  U∞ c / ν
ALPHA   = 4.0           # angle of attack [deg]
MACH    = 0.0           # freestream Mach number
NCRIT   = 9.0           # e^N transition criterion (9 = standard wind tunnel)

XFOIL_BIN = "../Xfoil/bin_nofpe/xfoil"

## Running XFOIL

In [ ]:
import os, re, shutil, subprocess, tempfile

def run_xfoil(airfoil, re_num, alpha, mach=0.0, ncrit=9.0, xfoil_bin=XFOIL_BIN):
    xfoil_bin = os.path.abspath(xfoil_bin)
    with tempfile.TemporaryDirectory() as tmp:
        if airfoil.upper().startswith("NACA"):
            load = f"NACA {airfoil.split()[-1]}"
        else:
            shutil.copy(airfoil, os.path.join(tmp, "airfoil.dat"))
            load = "LOAD airfoil.dat"

        cmds = "\n".join([
            "PLOP", "G", "",               # disable graphics
            load, "PANE",                  # load and repanel the airfoil
            "OPER",
            "VPAR", f"N {ncrit}", "",
            f"VISC {re_num}", f"MACH {mach}", "ITER 200",
            f"ALFA {alpha}",
            "CPWR cp.dat", "DUMP bl.dat",
            "", "QUIT", ""])

        env = {k: v for k, v in os.environ.items() if k != "DISPLAY"}
        out = subprocess.run([xfoil_bin], input=cmds, capture_output=True,
                             text=True, cwd=tmp, env=env).stdout
        if "Convergence failed" in out or not os.path.exists(os.path.join(tmp, "bl.dat")):
            raise RuntimeError("XFOIL did not converge for this operating point")

        cp = np.loadtxt(os.path.join(tmp, "cp.dat"), comments="#")
        with open(os.path.join(tmp, "bl.dat")) as f:
            rows = [l.split() for l in f if not l.startswith("#")]

    # Airfoil-surface rows have 12 columns; the wake rows that follow have fewer
    bl = np.array([[float(v) for v in r] for r in rows if len(r) == 12])

    def last(pattern):
        return float(re.findall(pattern, out)[-1])

    return dict(
        s=bl[:, 0], x=bl[:, 1], y=bl[:, 2], ue=bl[:, 3], cf=bl[:, 6], cp=cp[:, 1],
        CL=last(r"CL =\s*(-?[\d.]+)"),  CD=last(r"CD =\s*(-?[\d.]+)"),
        CDf=last(r"CDf =\s*(-?[\d.]+)"), CDp=last(r"CDp =\s*(-?[\d.]+)"),
        xtr_upper=last(r"Side 1 .*?transition at x/c =\s*([\d.]+)"),
        xtr_lower=last(r"Side 2 .*?transition at x/c =\s*([\d.]+)"),
    )

res = run_xfoil(AIRFOIL, RE, ALPHA, MACH, NCRIT)
print(f"{AIRFOIL}, Re = {RE:.1e}, alpha = {ALPHA} deg")
print(f"CL = {res['CL']:.4f}   CD = {res['CD']:.5f}  (CDf = {res['CDf']:.5f}, CDp = {res['CDp']:.5f})")
print(f"Transition: upper x/c = {res['xtr_upper']:.3f}, lower x/c = {res['xtr_lower']:.3f}")

## Splitting the surface at the stagnation point

In [ ]:
x, y, ue, cp, cf, s = (res[k] for k in ("x", "y", "ue", "cp", "cf", "s"))

i0 = np.argmax(ue < 0)                       # first point on the lower branch
w = ue[i0 - 1]/(ue[i0 - 1] - ue[i0])         # linear interpolation to Ue = 0
s_stag = s[i0 - 1] + w*(s[i0] - s[i0 - 1])
x_stag = x[i0 - 1] + w*(x[i0] - x[i0 - 1])
y_stag = y[i0 - 1] + w*(y[i0] - y[i0 - 1])

upper = np.arange(i0)[::-1]                  # stagnation point -> TE
lower = np.arange(i0, len(x))
s_from_stag = np.abs(s - s_stag)

print(f"Stagnation point at x/c = {x_stag:.4f}, y/c = {y_stag:.4f}")
print(f"Check Cp = 1 - (Ue/U∞)²: max deviation = {np.max(np.abs(cp - (1 - ue**2))):.1e}")

## Integrating the surface coefficients

In [ ]:
a = np.radians(ALPHA)
eD = np.array([np.cos(a), np.sin(a)])
eL = np.array([-np.sin(a), np.cos(a)])

dx, dy = np.diff(x), np.diff(y)
cp_m = 0.5*(cp[1:] + cp[:-1])
cf_m = 0.5*(cf[1:] + cf[:-1])
flow_dir = np.where(np.arange(len(dx)) < i0 - 1, -1.0, 1.0)   # upper panels: flow against point order
# The panel straddling the stagnation point carries almost no shear; it is assigned to the lower side

F_p = np.array([np.sum(-cp_m*dy), np.sum(cp_m*dx)])            # -∮ Cp n ds
F_f = np.array([np.sum(cf_m*flow_dir*dx), np.sum(cf_m*flow_dir*dy)])  # ∮ Cf t ds

CL_int  = (F_p + F_f) @ eL
CDp_int = F_p @ eD
CDf_int = F_f @ eD

from IPython.display import Markdown, display
display(Markdown(f'''
| Quantity | Integrated from $C_p$, $C_f$ | XFOIL | Difference |
|---|---|---|---|
| $C_l$ | {CL_int:.4f} | {res["CL"]:.4f} | {100*(CL_int/res["CL"] - 1):+.2f} % |
| $C_{{d,f}}$ (friction) | {CDf_int:.5f} | {res["CDf"]:.5f} | {100*(CDf_int/res["CDf"] - 1):+.2f} % |
| $C_{{d,p}}$ (pressure) | {CDp_int:.5f} | {res["CDp"]:.5f} | {100*(CDp_int/res["CDp"] - 1):+.1f} % |
| Friction part of $C_l$ | {F_f @ eL:+.5f} | | |
'''))

## Results

In [ ]:
c_up, c_lo = 'tab:blue', 'tab:red'
fig, axs = plt.subplots(3, 1, figsize=(9, 10), sharex=True,
                        gridspec_kw=dict(height_ratios=[3, 3, 1.2], hspace=0.08))

# Pressure coefficient
ax = axs[0]
ax.plot(x[upper], cp[upper], c_up, lw=1.8, label='upper surface')
ax.plot(x[lower], cp[lower], c_lo, lw=1.8, ls='--', label='lower surface')
ax.axhline(0, color='k', lw=0.6)
ax.axhline(1, color='0.5', lw=0.6, ls=':')
ax.text(0.99, 0.97, r'stagnation, $C_p = 1$', ha='right', va='bottom', fontsize=8, color='0.4')
ax.invert_yaxis()
ax.set_ylabel('$C_p$')
ax.set_title(f'{AIRFOIL},  $Re = {RE:.1e}$,  $\\alpha = {ALPHA:g}^\\circ$,  '
             f'$C_l = {res["CL"]:.3f}$,  $C_d = {res["CD"]:.4f}$', fontsize=11)
ax.legend(loc='upper right')

# Skin-friction coefficient
ax = axs[1]
ax.plot(x[upper], cf[upper], c_up, lw=1.8, label='upper surface')
ax.plot(x[lower], cf[lower], c_lo, lw=1.8, ls='--', label='lower surface')
re_s = RE*s_from_stag[upper][1:]
ax.plot(x[upper][1:], 0.664/np.sqrt(re_s), color='0.45', lw=1, ls=':', label='flat plate, laminar (Blasius)')
ax.plot(x[upper][1:], 0.0592*re_s**-0.2, color='0.45', lw=1, ls='-.', label='flat plate, turbulent')
ax.axhline(0, color='k', lw=0.6)
ax.set_ylim(min(-0.001, 1.2*cf.min()), 1.15*cf.max())
ax.set_ylabel('$C_f$')
ax.legend(loc='center right', fontsize=8)

for ax in axs[:2]:
    ax.axvline(res['xtr_upper'], color=c_up, lw=0.8, ls=':')
    ax.axvline(res['xtr_lower'], color=c_lo, lw=0.8, ls=':')
    ax.grid(alpha=0.25)
axs[1].text(res['xtr_upper'] + 0.01, 0.9*axs[1].get_ylim()[1], 'transition\n(upper)',
            color=c_up, fontsize=8, va='top')
axs[1].text(res['xtr_lower'] - 0.01, 0.9*axs[1].get_ylim()[1], 'transition\n(lower)',
            color=c_lo, fontsize=8, va='top', ha='right')

# Zoom on the trailing-edge region of the lower surface
ins = axs[1].inset_axes([0.40, 0.50, 0.28, 0.42])
zoom = lower[x[lower] > 0.8]
ins.plot(x[zoom], cf[zoom], c_lo, lw=1.5, ls='--')
ins.fill_between(x[zoom], cf[zoom], 0, where=cf[zoom] < 0, color=c_lo, alpha=0.3, label='$C_f < 0$')
ins.axhline(0, color='k', lw=0.6)
ins.axvline(res['xtr_lower'], color=c_lo, lw=0.8, ls=':')
ins.set_title('lower surface near the TE', fontsize=8)
ins.tick_params(labelsize=7)
ins.legend(fontsize=7, loc='upper left')

# Geometry
ax = axs[2]
ax.fill(x, y, color='0.88', ec='k', lw=1)
ax.plot(x_stag, y_stag, 'ko', ms=5, label='stagnation point')
for xt, idx, col in ((res['xtr_upper'], upper, c_up), (res['xtr_lower'], lower, c_lo)):
    ax.plot(xt, np.interp(xt, x[idx], y[idx]), 'x', color=col, ms=9, mew=2)
ax.plot([], [], 'kx', ms=8, mew=2, label='transition')
ax.set_aspect('equal')
ax.set_xlabel('$x/c$')
ax.set_yticks([])
ax.legend(loc='lower right', fontsize=8, ncol=2, frameon=False, bbox_to_anchor=(1.0, 0.0))
for side in ('top', 'right', 'left'):
    ax.spines[side].set_visible(False)

plt.show()